# 6-1. 품목코드 라벨링 — 전체 676,537행 유지 (검출률 0% 품목 삭제 안 함)

- 작성: 2026-09-28
- 목적: 노트북 6(`6_교수님과 미팅 후 품목 코드까지 조인.ipynb`)의 **6절 품목코드 라벨링**을, 같은 노트북 **2절 "검출률0% 품목 삭제"를 하지 않고** 원자료 전체 행에 적용한 뒤 `리스테리아ML/data/` 에 저장한다.

| | 노트북 6 | 이 노트북 (6-1) |
|---|---|---|
| 입력 | `data/listeria_body.parquet` 676,537행 × 23열 | 같음 |
| 검출이 한 건도 없는 학습용 품목명(`train_prdlst_nm`) 38개, 76,497행 | 삭제 → `df_filtered` 600,040행 | **삭제 안 함** → 676,537행 그대로 |
| 품목 단계 조인 | `ba_data.t_prdlst_cl` 을 `train_prdlst_code` 로 왼쪽 조인 | 같음 (같은 조인 논리) |
| 5절에서 만든 열(`is_processed`·`가공식품_여부`·`train_prdlst_code_str`·`has_C_prefix`) | 저장 파일에 들어 있음 | 넣지 않음 (5절 결과 설명 참고) |
| `train_prdlst_code` 형식 | 문자열(str)로 바꿔 저장 | 원자료 형식(category) 그대로 |
| 저장 파일 | `data/listeria_df_joined.parquet` (600,040행 × 41열) | **`data/listeria_df_joined_full.parquet`** (676,537행 × 40열 — 2026-10-01 의뢰 시스템 코드 3열 추가) |

- 원자료 23개 열은 값·행 순서를 바꾸지 않는다(행 삭제·결측 채움 없음). 조인으로 14개 열(`cl_kor_nm`, `cl_lv`, `lv1_cd`~`lv6_nm`)만 붙는다.
- 실행 방법: 위에서부터 차례로 실행하면 1~6절에서 단계별로 점검하고 **마지막 셀(8절)이 파일을 저장**한다. 마지막 셀은 앞 셀을 실행하지 않고 혼자 실행해도(새 커널) 같은 과정을 다시 거쳐 같은 파일을 만든다.
- DB: `ba_data.t_prdlst_cl` 을 SELECT 로 읽기만 한다. 세션은 읽기 전용(`default_transaction_read_only=on`)으로 열고, 비밀번호는 코드에 쓰지 않는다(`%APPDATA%\postgresql\pgpass.conf` 에서 libpq 가 읽음).
- 2026-10-01 추가: 7-1절에서 의뢰 시스템 코드(`reqest_sys_code`·`reqest_sys_nm`)와 수입 LIMS 연결 여부(`import_lims_linked`) 3개 열을 `lims_data_key` 로 붙인다. 저장 파일은 23 + 14 + 3 = 40열.

## 1. 원자료 불러오기 — 행을 지우지 않는다

In [1]:
import pandas as pd

df = pd.read_parquet('C:/Users/NT751/Desktop/천영진/리스테리아ML/data/listeria_body.parquet')

n_pos = int((df["judge_result"] == "검출").sum())
print(f"행 {len(df):,} / 열 {df.shape[1]} / 검출 {n_pos:,}건 (검출률 {n_pos / len(df) * 100:.3f}% = 검출 / 전체 행)")
display(df["judge_result"].value_counts(dropna=False).rename_axis("judge_result").reset_index(name="행수"))

행 676,537 / 열 23 / 검출 1,969건 (검출률 0.291% = 검출 / 전체 행)


,judge_result,행수
0,불검출,674568
1,검출,1969


## 2. 조인 키 점검 — `train_prdlst_code`(학습용 품목코드)

노트북 6 과 같이 조인 키는 품목명(`train_prdlst_nm`)이 아니라 코드(`train_prdlst_code`)다. 같은 한글명이 `ba_data.t_prdlst_cl` 에 코드 여러 개로 들어 있기 때문이다(예: 식육가공품 및 포장육 = C0202000000000·C0316000000000, 노트북 6 의 6절 [3] 출력). 이 노트북에서는 노트북 6 에서 삭제한 38개 품목도 조인 대상이다.

In [2]:
# 학습용 품목명·품목코드의 유니크 수와 결측
print(f"train_prdlst_nm   유니크 {df['train_prdlst_nm'].nunique():,}개 / 결측 {int(df['train_prdlst_nm'].isna().sum()):,}행")
print(f"train_prdlst_code 유니크 {df['train_prdlst_code'].nunique():,}개 / 결측 {int(df['train_prdlst_code'].isna().sum()):,}행")

# 코드 1개에 품목명이 1개씩인지 (코드 ↔ 품목명 1:1)
nm_per_code = df.dropna(subset=["train_prdlst_code"]).groupby("train_prdlst_code", observed=True)["train_prdlst_nm"].nunique()
print("품목명이 2개 이상 붙은 코드 수:", int((nm_per_code > 1).sum()))

# 코드가 비어 있는 행은 어떤 품목명·판정인가
na_code = df[df["train_prdlst_code"].isna()]
display(na_code.groupby(["train_prdlst_nm", "judge_result"], observed=True).size().rename("행수").reset_index())

train_prdlst_nm   유니크 53개 / 결측 0행
train_prdlst_code 유니크 52개 / 결측 170행
품목명이 2개 이상 붙은 코드 수: 0


,train_prdlst_nm,judge_result,행수
0,기타,불검출,170


**결과**
- 품목명 53개, 코드 52개. 코드 하나에 품목명이 하나씩 붙어 있다.
- 코드가 비어 있는 행은 `train_prdlst_nm` = '기타' 170행뿐이고 모두 불검출이다. 테이블정의서(`테이블명세서(모델팀)/테이블정의서.xlsx` ready_data 시트)의 `train_prdlst_nm` 설명은 "학습용 품목명. 미분류 시 '기타'(0.31%)"이다. 이 170행은 5절 조인에서 연결되지 않는다.

## 3. `ba_data.t_prdlst_cl` 내려받기 (읽기 전용 SELECT)

`ba_data.t_prdlst_cl` 은 DB 테이블 코멘트가 "식품코드"인 식품공전 품목분류 테이블이다(8,864행). 이 노트북에서 쓰는 컬럼과 DB 컬럼 코멘트(`col_description`)는 다음과 같다(아래 셀에서 DB 로 다시 확인).

| 컬럼 | DB 코멘트 | 이 노트북에서의 용도 |
|---|---|---|
| `prdlst_cd` | 품목_코드 | 조인 키 (`train_prdlst_code` 와 맞춤) |
| `kor_nm` | 한글_명 | 단계별 한글명 |
| `lv` | 레벨 | 코드의 단계 (1 = 최상위) |
| `hrnk_prdlst_cd` | 상위_품목_코드 | 한 단계 위 코드를 따라 올라갈 때 사용 |
| `htrk_prdlst_cd` | 최상위_품목_코드 | 참고용 (조인에는 쓰지 않음) |
| `use_yn`·`vald_begn_dt`·`vald_end_dt` | 사용_여부 · 유효_개시_일자 · `VALD_END_DT`(코멘트가 컬럼명 그대로 — 한글 설명 없음) | 6절 표에 참고로만 표시 (저장 파일에는 넣지 않음) |

In [3]:
# [1] DB 읽기 전용 접속 + ba_data.t_prdlst_cl 내려받기 — SELECT 만 (노트북 6 의 6절 [1] 과 같은 접속 방식)
import os
import psycopg

PGPASS = os.path.join(os.environ["APPDATA"], "postgresql", "pgpass.conf")   # 비밀번호는 libpq 가 이 파일에서 읽음
DB_HOST = "10.31.72.23"


def db_connect_readonly():
    """pgpass.conf 의 host 항목에서 port·dbname·user 만 읽어 읽기 전용 세션으로 접속한다."""
    with open(PGPASS, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.replace("\\:", "\x00").split(":")
            if len(parts) < 5:
                continue
            host, port, dbname, user = (x.replace("\x00", ":") for x in parts[:4])
            if host in (DB_HOST, "*"):
                conn = psycopg.connect(
                    host=DB_HOST, port=5432 if port == "*" else int(port), dbname=dbname, user=user,
                    options="-c default_transaction_read_only=on -c statement_timeout=120000",
                    application_name="listeria_nb6_1", connect_timeout=15)
                conn.autocommit = True
                return conn
    raise RuntimeError(f"pgpass.conf 에 {DB_HOST} 항목이 없습니다: {PGPASS}")


def db_query(sql, params=None):
    """SELECT 결과를 DataFrame 으로 (읽기 전용 세션, 실행마다 접속·종료)."""
    with db_connect_readonly() as conn, conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])


print("읽기 전용 세션:", db_query("show default_transaction_read_only").iloc[0, 0])

# 쓰는 컬럼의 DB 코멘트
display(db_query("""
    select a.attname as 컬럼, col_description(a.attrelid, a.attnum) as db_코멘트
    from pg_attribute a
    where a.attrelid = 'ba_data.t_prdlst_cl'::regclass and a.attnum > 0 and not a.attisdropped
      and a.attname in ('prdlst_cd', 'kor_nm', 'lv', 'hrnk_prdlst_cd', 'htrk_prdlst_cd', 'use_yn', 'vald_begn_dt', 'vald_end_dt')
    order by a.attnum
"""))

prdlst_cl = db_query("""
    select prdlst_cd, kor_nm, lv::int as lv, hrnk_prdlst_cd, htrk_prdlst_cd,
           prdlst_yn, use_yn, vald_begn_dt, vald_end_dt
    from ba_data.t_prdlst_cl
    order by prdlst_cd
""")
print(f"ba_data.t_prdlst_cl: {len(prdlst_cl):,}행 / prdlst_cd 고유 {prdlst_cl['prdlst_cd'].nunique():,}")
print("단계(lv)별 코드 수:", prdlst_cl["lv"].value_counts().sort_index().to_dict())

읽기 전용 세션: on


,컬럼,db_코멘트
0,prdlst_cd,품목_코드
1,kor_nm,한글_명
2,vald_begn_dt,유효_개시_일자
3,vald_end_dt,VALD_END_DT
4,lv,레벨
5,hrnk_prdlst_cd,상위_품목_코드
6,htrk_prdlst_cd,최상위_품목_코드
7,use_yn,사용_여부


ba_data.t_prdlst_cl: 8,864행 / prdlst_cd 고유 8,864
단계(lv)별 코드 수: {1: 9, 2: 106, 3: 3906, 4: 2029, 5: 2570, 6: 244}


## 4. 단계 전개 — 코드마다 상위 코드(`hrnk_prdlst_cd`)를 따라 1단계(최상위)까지

노트북 6 의 6절 [2] 와 같은 방법이다. 결과는 코드마다 `lv1_cd`/`lv1_nm` … `lv6_cd`/`lv6_nm` 열이 된다(해당 단계가 없으면 비어 있음).

In [4]:
# [2] 계층 전개: 코드마다 상위 코드(hrnk_prdlst_cd)를 따라 1단계(최상위)까지 올라가 단계별 코드·한글명 컬럼을 만든다
parent = dict(zip(prdlst_cl["prdlst_cd"], prdlst_cl["hrnk_prdlst_cd"]))
kor    = dict(zip(prdlst_cl["prdlst_cd"], prdlst_cl["kor_nm"]))
level  = dict(zip(prdlst_cl["prdlst_cd"], prdlst_cl["lv"]))
MAX_LV = int(prdlst_cl["lv"].max())


def code_chain(code):
    """code 에서 상위 코드를 따라 최상위까지: {1: 최상위코드, 2: ..., lv: code}. 상위 코드가 ba_data.t_prdlst_cl 에 없으면 거기서 멈춘다."""
    chain, seen = {}, set()
    while code in level and code not in seen:
        seen.add(code)
        chain[level[code]] = code
        p = parent.get(code)
        if not p or p == code:
            break
        code = p
    return chain


rows = []
for code in prdlst_cl["prdlst_cd"]:
    ch = code_chain(code)
    rec = {"prdlst_cd": code}
    for lv in range(1, MAX_LV + 1):
        rec[f"lv{lv}_cd"] = ch.get(lv)
        rec[f"lv{lv}_nm"] = kor.get(ch[lv]) if lv in ch else None
    rec["chain_top_lv"] = min(ch) if ch else None   # 체인이 실제로 닿은 가장 위 단계 (1 이면 최상위까지 완결)
    rows.append(rec)
prdlst_hier = prdlst_cl.merge(pd.DataFrame(rows), on="prdlst_cd")

# 원자료의 52개 코드가 ba_data.t_prdlst_cl 에 있는지, 1단계까지 닿는지
codes_in_df = pd.Series(df["train_prdlst_code"].dropna().astype(str).unique(), name="prdlst_cd")
chk = prdlst_hier[prdlst_hier["prdlst_cd"].isin(codes_in_df)]
print(f"원자료 코드 {len(codes_in_df)}개 중 ba_data.t_prdlst_cl 에 있음 {len(chk)}개, "
      f"1단계까지 닿음 {int((chk['chain_top_lv'] == 1).sum())}개")
print("1단계(최상위) 코드:")
display(prdlst_hier.loc[prdlst_hier["lv"] == 1, ["prdlst_cd", "kor_nm"]].reset_index(drop=True))

원자료 코드 52개 중 ba_data.t_prdlst_cl 에 있음 52개, 1단계까지 닿음 52개
1단계(최상위) 코드:


,prdlst_cd,kor_nm
0,A0000000000000,식물성
1,B0000000000000,동물성
2,C0000000000000,가공식품
3,D0000000000000,첨가물
4,E0000000000000,건강기능식품
5,F0000000000000,기구 및 용기포장
6,G0000000000000,식품접객업소
7,H0000000000000,위생용품
8,Z0000000000000,기준규격외


## 5. 전체 676,537행에 조인

- 왼쪽 조인(`how="left"`), `validate="many_to_one"`(ba_data.t_prdlst_cl 쪽 코드가 1행씩인지 확인).
- 원자료 `train_prdlst_code` 는 바꾸지 않는다. 문자열로 바꾼 임시 키 `_join_key` 로 붙이고 조인 뒤 지운다.
- 코드가 없는 행은 조인되지 않은 채(단계 열 결측)로 둔다. 채우지 않는다.

In [5]:
# [3] 전체 행(df)에 조인 — 키는 train_prdlst_code (train_prdlst_nm 은 ba_data.t_prdlst_cl 에서 이름이 중복되므로 키로 쓰지 않음)
hier_cols = ["prdlst_cd", "kor_nm", "lv"] + [f"lv{l}_{k}" for l in range(1, MAX_LV + 1) for k in ("cd", "nm")]
join_tbl = (prdlst_hier[hier_cols]
            .rename(columns={"prdlst_cd": "_join_key", "kor_nm": "cl_kor_nm", "lv": "cl_lv"}))
join_tbl["_join_key"] = join_tbl["_join_key"].astype(str)

df_joined = (df.assign(_join_key=df["train_prdlst_code"].astype(str))   # 임시 키 (원자료 열은 그대로)
               .merge(join_tbl, on="_join_key", how="left", validate="many_to_one")
               .drop(columns="_join_key"))
df_joined["cl_lv"] = df_joined["cl_lv"].astype("Int64")   # 조인 안 된 행은 <NA> (소수점 2.0 대신 2)

matched = df_joined["cl_lv"].notna()
print(f"조인 전 {len(df):,}행 → 조인 후 {len(df_joined):,}행 (같음: {len(df_joined) == len(df)})")
print(f"원자료 23개 열의 값·행 순서 그대로: {df_joined[df.columns].equals(df)}")
print(f"코드 매칭 {int(matched.sum()):,}행 ({matched.mean() * 100:.3f}%) / 매칭 안 됨 {int((~matched).sum()):,}행")
name_same = df_joined.loc[matched, "train_prdlst_nm"].astype(str) == df_joined.loc[matched, "cl_kor_nm"].astype(str)
print(f"매칭된 행에서 train_prdlst_nm 과 ba_data.t_prdlst_cl.kor_nm 이 같은 비율: {name_same.mean() * 100:.2f}%")
print(f"열: 원자료 {df.shape[1]}개 + 조인 {df_joined.shape[1] - df.shape[1]}개 = {df_joined.shape[1]}개")

# 매칭 안 된 행: 원본 품목코드(prdlst_code) 앞 14자리가 ba_data.t_prdlst_cl 에 있는지
um = df_joined.loc[~matched, ["train_prdlst_nm", "prdlst_code", "judge_result"]].copy()
um["prdlst_code_14"] = um["prdlst_code"].astype(str).str[:14]
um["t_prdlst_cl_에_있음"] = um["prdlst_code_14"].isin(prdlst_cl["prdlst_cd"])
print("\n매칭 안 된 행 — 원본 품목코드(prdlst_code)별 행 수:")
display(um.groupby(["train_prdlst_nm", "prdlst_code_14", "t_prdlst_cl_에_있음", "judge_result"], observed=True)
          .size().rename("행수").reset_index().sort_values("행수", ascending=False).reset_index(drop=True))

조인 전 676,537행 → 조인 후 676,537행 (같음: True)
원자료 23개 열의 값·행 순서 그대로: True
코드 매칭 676,367행 (99.975%) / 매칭 안 됨 170행
매칭된 행에서 train_prdlst_nm 과 ba_data.t_prdlst_cl.kor_nm 이 같은 비율: 100.00%
열: 원자료 23개 + 조인 14개 = 37개



매칭 안 된 행 — 원본 품목코드(prdlst_code)별 행 수:


,train_prdlst_nm,prdlst_code_14,t_prdlst_cl_에_있음,judge_result,행수
0,기타,C0111020000000,False,불검출,126
1,기타,A1213000000000,False,불검출,25
2,기타,C0130070200000,False,불검출,8
3,기타,C0111030000000,False,불검출,4
4,기타,E03ZXV00000000,False,불검출,4
5,기타,A1237000000000,False,불검출,1
6,기타,C0130010100000,False,불검출,1
7,기타,E0344500000000,False,불검출,1


In [6]:
# 코드 첫 글자와 최상위(lv1_cd) 첫 글자가 다른 코드 — 노트북 6 의 5절 "코드가 C 로 시작하면 가공식품" 규칙과 비교
first = df_joined.loc[matched, ["train_prdlst_code", "train_prdlst_nm", "cl_lv", "lv1_cd", "lv1_nm"]].copy()
first["train_prdlst_code"] = first["train_prdlst_code"].astype(str)
diff = first[first["train_prdlst_code"].str[0] != first["lv1_cd"].str[0]]
display(diff.groupby(["train_prdlst_code", "train_prdlst_nm", "cl_lv", "lv1_cd", "lv1_nm"], observed=True)
            .size().rename("행수").reset_index())

c_prefix = df_joined["train_prdlst_code"].astype(str).str.startswith("C", na=False)   # 노트북 6 의 5절 is_processed 규칙
lv1_c = df_joined["lv1_cd"].eq("C0000000000000")
print(f"'코드가 C 로 시작' {int(c_prefix.sum()):,}행 / 'lv1_cd = C0000000000000(가공식품)' {int(lv1_c.sum()):,}행 / "
      f"두 기준이 다른 행 {int((c_prefix != lv1_c).sum()):,}행")

,train_prdlst_code,train_prdlst_nm,cl_lv,lv1_cd,lv1_nm,행수
0,X0100027500000,기타제조가공식품류(자가규격),2,C0000000000000,가공식품,342
1,X0100027900000,환경오염도조사용,2,G0000000000000,식품접객업소,14


'코드가 C 로 시작' 332,934행 / 'lv1_cd = C0000000000000(가공식품)' 333,276행 / 두 기준이 다른 행 342행


**결과**
- 조인 뒤에도 676,537행 그대로이고, 원자료 23개 열의 값과 행 순서는 바뀌지 않았다. 열은 23 + 14 = 37개.
- 코드 매칭 676,367행(99.975%). 매칭된 행은 `train_prdlst_nm` 과 `ba_data.t_prdlst_cl.kor_nm` 이 100% 같다.
- 매칭 안 된 170행은 모두 '기타'다. 이 행들의 원본 품목코드 `prdlst_code` 앞 14자리(C0111020000000 126행, A1213000000000 25행 등 8종)가 `ba_data.t_prdlst_cl` 에 없다. 단계 열(`cl_kor_nm`·`cl_lv`·`lv1_cd`~`lv6_nm`)은 비워 두었다(채우지 않음). **이 170행을 모델에서 그대로 둘지·뺄지·따로 묶을지는 정해야 한다.**
- 코드 첫 글자와 최상위 분류가 다른 코드가 2개 있다(`ba_data.t_prdlst_cl.hrnk_prdlst_cd` 기준).
  - `X0100027500000` 기타제조가공식품류(자가규격) 342행 → 상위 코드 C0000000000000(가공식품)
  - `X0100027900000` 환경오염도조사용 14행 → 상위 코드 G0000000000000(식품접객업소)
- 그래서 노트북 6 의 5절 규칙("`train_prdlst_code` 가 C 로 시작하면 가공식품", `is_processed`)과 "`lv1_cd` 가 가공식품"인 행이 전체 행에서는 342행 다르다. 노트북 6 의 600,040행에서는 이 코드가 삭제돼 있어 차이가 0이었다. 이 때문에 5절 열 4개(`is_processed`·`가공식품_여부`·`train_prdlst_code_str`·`has_C_prefix`)는 저장 파일에 넣지 않았다. 가공식품 여부가 필요하면 `lv1_cd == 'C0000000000000'` 또는 코드 첫 글자 중 어느 기준을 쓸지 정한 뒤 만든다.

## 6. 결과 요약 — 품목별 단계·경로와 최상위 분류별 검출

- 검출률(%) = 그 그룹의 검출 / 그 그룹의 행 수 × 100
- 검출비중(%) = 그 그룹의 검출 / 전체 검출 1,969건 × 100
- `노트북6_삭제`: 노트북 6 의 2절 규칙(검출이 한 건도 없는 품목명 삭제)에 걸리는 품목 = True. 이 노트북에서는 지우지 않고 표시만 한다.

In [7]:
# [4] 품목명 53개(코드 52개 + 코드 없는 '기타')의 단계·경로·행 수·검출 (전체 676,537행 기준)
n_pos_all = int((df_joined["judge_result"] == "검출").sum())
lvl_map = (df_joined.groupby(["train_prdlst_code", "train_prdlst_nm"], observed=True, dropna=False)
           .agg(행수=("judge_result", "size"), 검출=("judge_result", lambda s: int((s == "검출").sum())))
           .reset_index())
lvl_map = lvl_map.assign(_join_key=lvl_map["train_prdlst_code"].astype(str)).merge(
    join_tbl.merge(prdlst_cl[["prdlst_cd", "use_yn", "vald_end_dt"]].rename(columns={"prdlst_cd": "_join_key"}), on="_join_key"),
    on="_join_key", how="left").drop(columns="_join_key")
lvl_map["노트북6_삭제"] = lvl_map["검출"] == 0
lvl_map["검출률(%)"]   = (lvl_map["검출"] / lvl_map["행수"] * 100).round(3)
lvl_map["검출비중(%)"] = (lvl_map["검출"] / n_pos_all * 100).round(2)
lvl_map["cl_lv"] = lvl_map["cl_lv"].astype("Int64")
lvl_map["계층 경로(한글명)"] = lvl_map.apply(
    lambda r: " > ".join(str(r[f"lv{l}_nm"]) for l in range(1, MAX_LV + 1) if pd.notna(r[f"lv{l}_nm"]))
              or "(train_prdlst_code 없음 — 조인 안 됨)", axis=1)
lvl_map = lvl_map.sort_values(["lv1_cd", "cl_lv", "train_prdlst_code"], na_position="last").reset_index(drop=True)

show_cols = ["train_prdlst_code", "train_prdlst_nm", "노트북6_삭제", "cl_lv", "lv1_nm", "계층 경로(한글명)",
             "행수", "검출", "검출률(%)", "검출비중(%)", "use_yn", "vald_end_dt"]
with pd.option_context("display.max_rows", 100, "display.max_colwidth", 60):
    display(lvl_map[show_cols])
print(f"품목명 {len(lvl_map)}개 (노트북 6 에서 삭제되는 품목 {int(lvl_map['노트북6_삭제'].sum())}개, "
      f"{int(lvl_map.loc[lvl_map['노트북6_삭제'], '행수'].sum()):,}행)")
print("품목 코드가 속한 단계(cl_lv)별 품목 수:", lvl_map["cl_lv"].value_counts(dropna=False).sort_index().to_dict())

,train_prdlst_code,train_prdlst_nm,노트북6_삭제,cl_lv,lv1_nm,계층 경로(한글명),행수,검출,검출률(%),검출비중(%),use_yn,vald_end_dt
0,A0100000000000,곡류,True,2,식물성,식물성 > 곡류,52,0,0.000,0.00,Y,99991231
1,A0200000000000,서류,True,2,식물성,식물성 > 서류,4,0,0.000,0.00,Y,99991231
2,A0300000000000,두류,True,2,식물성,식물성 > 두류,8,0,0.000,0.00,Y,99991231
3,A0400000000000,견과 종실류,False,2,식물성,식물성 > 견과 종실류,292,1,0.342,0.05,Y,99991231
4,A0500000000000,과일류,True,2,식물성,식물성 > 과일류,702,0,0.000,0.00,Y,99991231
5,A0600000000000,채소류,True,2,식물성,식물성 > 채소류,904,0,0.000,0.00,Y,99991231
6,A0700000000000,버섯류,False,2,식물성,식물성 > 버섯류,92,12,13.043,0.61,Y,99991231
7,A1100000000000,조류,True,2,식물성,식물성 > 조류,1,0,0.000,0.00,Y,99991231
8,A1200000000000,기타식물류,False,2,식물성,식물성 > 기타식물류,43,2,4.651,0.10,Y,99991231
9,B0100000000000,축산물,False,2,동물성,동물성 > 축산물,1731,28,1.618,1.42,Y,99991231


품목명 53개 (노트북 6 에서 삭제되는 품목 38개, 76,497행)
품목 코드가 속한 단계(cl_lv)별 품목 수: {np.int64(2): 28, np.int64(3): 24, <NA>: 1}


In [8]:
# [5] 최상위 분류(lv1_nm)별 행 수·검출 — 전체(이 노트북) vs 노트북 6(검출 0% 품목 삭제 후 600,040행)
kept = df_joined["train_prdlst_nm"].isin(lvl_map.loc[~lvl_map["노트북6_삭제"], "train_prdlst_nm"])   # 노트북 6 df_filtered 에 남는 행


def lv1_summary(d):
    return (d.groupby(["lv1_cd", "lv1_nm"], dropna=False)
             .agg(행수=("judge_result", "size"), 검출=("judge_result", lambda s: int((s == "검출").sum()))))


top = lv1_summary(df_joined).join(lv1_summary(df_joined[kept]), rsuffix="_노트북6", how="left")
top[["행수_노트북6", "검출_노트북6"]] = top[["행수_노트북6", "검출_노트북6"]].fillna(0).astype(int)   # 노트북 6 에 없는 분류 = 0행
top["검출률(%)"]   = (top["검출"] / top["행수"] * 100).round(3)
top["검출비중(%)"] = (top["검출"] / n_pos_all * 100).round(1)
top["늘어난 행수"] = top["행수"] - top["행수_노트북6"]
top = top.reset_index().sort_values(["검출", "행수"], ascending=False).reset_index(drop=True)
top["lv1_nm"] = top["lv1_nm"].fillna("(train_prdlst_code 없음 — 조인 안 됨)")   # 표시용 이름 (저장 데이터는 결측 그대로)
top.loc[len(top)] = ["(합계)", "", top["행수"].sum(), top["검출"].sum(), top["행수_노트북6"].sum(), top["검출_노트북6"].sum(),
                     round(top["검출"].sum() / top["행수"].sum() * 100, 3), 100.0, top["늘어난 행수"].sum()]
display(top[["lv1_cd", "lv1_nm", "행수", "검출", "검출률(%)", "검출비중(%)", "행수_노트북6", "늘어난 행수"]])

,lv1_cd,lv1_nm,행수,검출,검출률(%),검출비중(%),행수_노트북6,늘어난 행수
0,Z0000000000000,기준규격외,320550,1431,0.446,72.7,320464,86
1,C0000000000000,가공식품,333276,449,0.135,22.8,268945,64331
2,B0000000000000,동물성,10204,74,0.725,3.8,10204,0
3,A0000000000000,식물성,2098,15,0.715,0.8,427,1671
4,G0000000000000,식품접객업소,9654,0,0.000,0.0,0,9654
5,E0000000000000,건강기능식품,446,0,0.000,0.0,0,446
6,NaN,(train_prdlst_code 없음 — 조인 안 됨),170,0,0.000,0.0,0,170
7,D0000000000000,첨가물,139,0,0.000,0.0,0,139
8,(합계),,676537,1969,0.291,100.0,600040,76497


**결과** (검출 1,969건은 노트북 6 과 같다. 늘어난 76,497행은 모두 불검출)
- 전체 검출률: 0.328%(노트북 6, 600,040행) → **0.291%**(676,537행).
- 최상위 분류(`lv1_nm`)가 노트북 6 의 4개(식물성·동물성·가공식품·기준규격외)에서 7개로 늘고, 코드가 없는 '기타' 170행이 따로 남는다. 새로 들어온 분류는 식품접객업소(G) 9,654행, 건강기능식품(E) 446행, 첨가물(D) 139행이며 검출은 모두 0이다.
- 가공식품(C): 268,945 → 333,276행(+64,331행: 알가공품류 29,392 · 빙과류 22,206 · 조미식품 4,165 등). 검출률 0.167% → 0.135%.
- 식물성(A): 427 → 2,098행(+1,671행: 채소류 904 · 과일류 702 등). 검출률 3.513% → 0.715%.
- 기준규격외(Z): +86행(환경검체 35 · 수산가공 40 · 농산물 종자류 8 · 인삼.홍삼제품류 3). 동물성(B)은 변화 없음(38개 삭제 품목에 B 코드가 없음).
- 품목 코드 52개의 단계(`cl_lv`): 2단계 28개, 3단계 24개(C03xx). 4단계 이하 없음.
- 참고: `D0600000000000` 식품첨가물(70행)은 `ba_data.t_prdlst_cl.vald_end_dt` = 20260414 이고, 나머지 51개 코드는 99991231 이다. `vald_end_dt` 는 DB 코멘트가 컬럼명 그대로라 뜻을 확인하지 못했다 — 확인 필요.

## 7. 노트북 6 저장 파일과 대조

이 노트북 결과에서 노트북 6 이 남긴 15개 품목의 행만 골라 `data/listeria_df_joined.parquet`(노트북 6 저장본, 600,040행)과 열마다 값을 비교한다. 같다면 이 노트북 결과는 "노트북 6 결과 + 삭제됐던 76,497행"이다.

**2026-10-01 안내**: 노트북 6 이 검출 0% 품목을 지우지 않게 바뀌어 `data/listeria_df_joined.parquet` 도 676,537행 × 41열이 됐다. 아래 셀의 출력과 결과 문구는 그 전(600,040행 저장본)에 실행한 것이다. 이 셀을 다시 실행하면 행 수가 달라 '값이 다른 열'로 나오므로, 600,040행 대조는 더 이상 성립하지 않는다.

In [9]:
from pathlib import Path

NB6_PATH = Path(r"C:\Users\NT751\Desktop\천영진\리스테리아ML\data\listeria_df_joined.parquet")


def same_values(a, b):
    """두 열의 결측 위치와 값(문자열로 비교)이 같은지 — category/str, Int64/int64 처럼 형식만 다른 것은 같게 본다."""
    a, b = a.reset_index(drop=True), b.reset_index(drop=True)
    if len(a) != len(b) or not a.isna().equals(b.isna()):
        return False
    return bool((a[a.notna()].astype(str).to_numpy() == b[b.notna()].astype(str).to_numpy()).all())


if NB6_PATH.exists():
    nb6 = pd.read_parquet(NB6_PATH)
    sub = df_joined[kept].reset_index(drop=True)
    common = [c for c in df_joined.columns if c in nb6.columns]
    diff_cols = [c for c in common if not same_values(sub[c], nb6[c])]
    print(f"노트북 6 저장본: {len(nb6):,}행 × {nb6.shape[1]}열 / 이 노트북에서 같은 15개 품목만: {len(sub):,}행")
    print(f"공통 열 {len(common)}개 중 값이 다른 열: {diff_cols if diff_cols else '없음'}")
    print("노트북 6 저장본에만 있는 열:", [c for c in nb6.columns if c not in df_joined.columns])
    print("형식(dtype)이 다른 공통 열:", {c: (str(sub[c].dtype), str(nb6[c].dtype)) for c in common if sub[c].dtype != nb6[c].dtype})
else:
    print("노트북 6 저장본이 없어 대조를 건너뜀:", NB6_PATH)

노트북 6 저장본: 600,040행 × 41열 / 이 노트북에서 같은 15개 품목만: 600,040행
공통 열 37개 중 값이 다른 열: 없음
노트북 6 저장본에만 있는 열: ['is_processed', '가공식품_여부', 'train_prdlst_code_str', 'has_C_prefix']
형식(dtype)이 다른 공통 열: {'train_prdlst_code': ('category', 'str'), 'cl_lv': ('Int64', 'int64')}


**결과**: 노트북 6 이 남긴 15개 품목의 600,040행만 고르면, 공통 37개 열의 값이 노트북 6 저장본과 전부 같다(행 순서 포함). 형식만 다른 열은 `train_prdlst_code`(이 노트북은 원자료 그대로 category, 노트북 6 은 str)와 `cl_lv`(이 노트북은 조인 안 된 행을 비워 두려고 Int64, 노트북 6 은 int64) 두 개다. 따라서 이 노트북의 저장 파일은 "노트북 6 저장본에서 5절 열 4개를 뺀 것 + 삭제됐던 76,497행"이다.

## 7-1. 의뢰 시스템 코드 조인 (2026-10-01 추가)

수입 검사로 들어온 행과 국내 의뢰 검사 행을 가르는 값을 붙인다. `src_route`(= DB `geo_route`, 테이블정의서 `ready_data` 시트: "domestic(한글주소→법정동) / overseas(비한글→지오코딩)")는 주소에 한글이 있는지로 갈린 값이라 수입 여부와 221행이 어긋난다(작업기록 2026-10-01).

| 붙이는 열 | 어디서 | 설명 문구 |
|---|---|---|
| `reqest_sys_code` | `ba_data.m_org_lims_data_info.reqest_sys_code` | DB 코멘트 "ZCI07 의뢰 시스템 코드" |
| `reqest_sys_nm` | `ba_data.t_cm_cmmn_code.cmmn_code_nm` (`cmmn_cl_code` = 'CI07', `cmmn_code` = `reqest_sys_code`) | 코드 이름. 값: 자체의뢰·통합망·수입축산물·수입수산물·수입식품 |
| `import_lims_linked` | `ready_data.hazard_preprocessed_data.cntc_sys_key2` 가 `ba_data.s_org_lims_data_info.rcept_no`(DB 코멘트 "접수번호", `label` = '리스테리아')에 있으면 True | 수입식품 LIMS(`ba_data.s_org_lims_data_info`, DB 테이블 코멘트 "수입식품 LIMS 검사 결과")에 연결되는 행 |

- 조인 키는 행 키 `lims_data_key`(`ready_data.hazard_preprocessed_data` 리스테리아 765,891행에서 받아 왼쪽 조인). 읽기 전용 세션에서 SELECT 만 한다.
- `ba_data.m_org_lims_data_info` 에 없는 행은 `reqest_sys_code`·`reqest_sys_nm` 이 비어 있다(채우지 않음). 이 행들은 `import_lims_linked` 로 구분된다.
- 이 열들이 가르는 것은 "어느 의뢰 시스템으로 들어온 검사인가"이다. 주소가 어느 나라인지, 제품이 어디서 만들어졌는지를 뜻하지 않는다.

In [10]:
# [6] 의뢰 시스템 코드·수입 LIMS 연결 여부 — 읽기 전용 SELECT, lims_data_key 로 왼쪽 조인
SQL_REQEST_SYS = """
        select h.lims_data_key,
               m.reqest_sys_code,
               c.cmmn_code_nm as reqest_sys_nm,
               (s.rcept_no is not null) as import_lims_linked
        from ready_data.hazard_preprocessed_data h
        left join ba_data.m_org_lims_data_info m on m.lims_data_key = h.lims_data_key
        left join ba_data.t_cm_cmmn_code c on c.cmmn_cl_code = 'CI07' and c.cmmn_code = m.reqest_sys_code
        left join (select distinct rcept_no from ba_data.s_org_lims_data_info where label = '리스테리아') s
               on s.rcept_no = h.cntc_sys_key2
        where h.hazard_label = '리스테리아'
"""
display(db_query("""
    select 'ba_data.m_org_lims_data_info.reqest_sys_code' as 컬럼, col_description('ba_data.m_org_lims_data_info'::regclass, a.attnum) as db_코멘트
    from pg_attribute a where a.attrelid = 'ba_data.m_org_lims_data_info'::regclass and a.attname = 'reqest_sys_code'
    union all
    select 'ba_data.s_org_lims_data_info.rcept_no', col_description('ba_data.s_org_lims_data_info'::regclass, a.attnum)
    from pg_attribute a where a.attrelid = 'ba_data.s_org_lims_data_info'::regclass and a.attname = 'rcept_no'
"""))
reqest_sys = db_query(SQL_REQEST_SYS)
assert reqest_sys["lims_data_key"].is_unique, "lims_data_key 가 겹침 — 조인하면 행이 늘어난다"
print(f"DB 에서 받은 행 {len(reqest_sys):,} (ready_data.hazard_preprocessed_data 리스테리아 행)")

# 원자료 열(lims_data_key 포함)의 형식을 건드리지 않도록, 키 순서에 맞춘 값만 새 열로 붙인다
_aligned = reqest_sys.set_index("lims_data_key").reindex(df_joined["lims_data_key"].astype(str).to_numpy())
assert _aligned["import_lims_linked"].notna().all(), "DB 에 없는 lims_data_key 가 있음"
df_joined["reqest_sys_code"] = _aligned["reqest_sys_code"].to_numpy()
df_joined["reqest_sys_nm"] = _aligned["reqest_sys_nm"].to_numpy()
df_joined["import_lims_linked"] = _aligned["import_lims_linked"].to_numpy().astype(bool)
print(f"조인 후 {len(df_joined):,}행 × {df_joined.shape[1]}열")

# 의뢰 시스템 코드 이름 × 수입 LIMS 연결 × src_route — 행 수와 검출
chk = (df_joined.assign(검출=(df_joined["judge_result"] == "검출").astype(int),
                        reqest_sys_nm=df_joined["reqest_sys_nm"].fillna("(m_org_lims_data_info 에 없음)"))
       .groupby(["reqest_sys_code", "reqest_sys_nm", "import_lims_linked", "src_route"], observed=True, dropna=False)
       .agg(행수=("검출", "size"), 검출=("검출", "sum")).reset_index())
display(chk)
is_import_sys = df_joined["reqest_sys_nm"].astype("string").str.startswith("수입").fillna(False) | df_joined["reqest_sys_nm"].isna()
print("의뢰 시스템 코드 이름이 '수입…' 이거나 m_org_lims_data_info 에 없는 행 = 수입 LIMS 연결 행인가:",
      bool((is_import_sys == df_joined["import_lims_linked"]).all()))
print(f"수입 LIMS 연결 {int(df_joined['import_lims_linked'].sum()):,}행 / 연결 없음 {int((~df_joined['import_lims_linked']).sum()):,}행")
print("src_route 와 어긋나는 행:",
      f"overseas 인데 연결 없음 {int(((df_joined['src_route'] == 'overseas') & ~df_joined['import_lims_linked']).sum()):,} ·",
      f"domestic 인데 연결됨 {int(((df_joined['src_route'] == 'domestic') & df_joined['import_lims_linked']).sum()):,}")

,컬럼,db_코멘트
0,ba_data.m_org_lims_data_info.reqest_sys_code,ZCI07 의뢰 시스템 코드
1,ba_data.s_org_lims_data_info.rcept_no,접수번호


DB 에서 받은 행 765,891 (ready_data.hazard_preprocessed_data 리스테리아 행)


조인 후 676,537행 × 40열


,reqest_sys_code,reqest_sys_nm,import_lims_linked,src_route,행수,검출
0,CI07000001,자체의뢰,False,domestic,593744,1780
1,CI07000001,자체의뢰,False,overseas,211,0
2,CI07000002,수입축산물,True,domestic,10,0
3,CI07000002,수입축산물,True,overseas,31573,73
4,CI07000003,수입수산물,True,overseas,7279,42
5,CI07000004,수입식품,True,overseas,1376,0
6,CI07000006,통합망,False,domestic,35141,64
7,NaN,(m_org_lims_data_info 에 없음),True,overseas,7203,10


의뢰 시스템 코드 이름이 '수입…' 이거나 m_org_lims_data_info 에 없는 행 = 수입 LIMS 연결 행인가: True
수입 LIMS 연결 47,441행 / 연결 없음 629,096행
src_route 와 어긋나는 행: overseas 인데 연결 없음 211 · domestic 인데 연결됨 10


**결과 (2026-10-01 실행)**
- 의뢰 시스템 코드 이름이 수입축산물·수입수산물·수입식품인 행과 `ba_data.m_org_lims_data_info` 에 없는 행은 전부 수입 LIMS 에 연결되고, 자체의뢰·통합망 행은 전부 연결되지 않는다(위 출력). 수치는 위 표.
- `src_route` 와 어긋나는 행: overseas 인데 수입 LIMS 연결이 없는 211행(주소가 로마자로 적힌 자체의뢰 행), domestic 인데 연결되는 10행.

## 8. 저장 — `data/listeria_df_joined_full.parquet`

- 이 셀 하나만 실행해도(새 커널, 앞 셀 실행 없이) 1·3·4·5절과 같은 과정(원자료 읽기 → `ba_data.t_prdlst_cl` SELECT → 단계 전개 → 왼쪽 조인 → 의뢰 시스템 코드 조인)을 거쳐 같은 파일을 저장한다. 같은 이름의 파일이 있으면 덮어쓴다.
- 앞 절을 실행한 뒤라면 단계별 결과(`df_joined`)와 같은지도 출력한다.
- 저장 내용: 원자료 23개 열(값·행 순서 그대로) + `cl_kor_nm`(`train_prdlst_code` 의 `ba_data.t_prdlst_cl.kor_nm`) + `cl_lv`(그 코드의 `lv`) + `lv1_cd`~`lv6_nm`(1~6단계 코드·한글명) + 의뢰 시스템 코드 3개 열(`reqest_sys_code`·`reqest_sys_nm`·`import_lims_linked`, 7-1절) = 40개 열.
- `ba_data.t_prdlst_cl` 내용이 바뀌면 단계 열 값도 바뀔 수 있다(2026-09-28 실행 기준 8,864행).

In [11]:
# [저장] data/listeria_body.parquet 676,537행(검출률 0% 품목 삭제 안 함) + ba_data.t_prdlst_cl 단계(lv1~lv6) 왼쪽 조인
#        + 의뢰 시스템 코드 3열(7-1절) → data/listeria_df_joined_full.parquet
# 앞 셀을 실행하지 않고 이 셀만 실행해도(새 커널) 같은 과정으로 같은 파일을 만든다.
import os
from pathlib import Path

import pandas as pd
import psycopg

IN_PATH  = "C:/Users/NT751/Desktop/천영진/리스테리아ML/data/listeria_body.parquet"
OUT_PATH = Path(r"C:\Users\NT751\Desktop\천영진\리스테리아ML\data\listeria_df_joined_full.parquet")
PGPASS   = os.path.join(os.environ["APPDATA"], "postgresql", "pgpass.conf")   # 비밀번호는 libpq 가 이 파일에서 읽음
DB_HOST  = "10.31.72.23"


def _connect_readonly():
    """pgpass.conf 의 host 항목에서 port·dbname·user 만 읽어 읽기 전용 세션으로 접속한다 (SELECT 만 가능)."""
    with open(PGPASS, encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.replace("\\:", "\x00").split(":")
            if len(parts) < 5:
                continue
            host, port, dbname, user = (x.replace("\x00", ":") for x in parts[:4])
            if host in (DB_HOST, "*"):
                conn = psycopg.connect(
                    host=DB_HOST, port=5432 if port == "*" else int(port), dbname=dbname, user=user,
                    options="-c default_transaction_read_only=on -c statement_timeout=120000",
                    application_name="listeria_nb6_1", connect_timeout=15)
                conn.autocommit = True
                return conn
    raise RuntimeError(f"pgpass.conf 에 {DB_HOST} 항목이 없습니다: {PGPASS}")


# 1) 원자료 — 행 삭제·결측 채움 없음
src = pd.read_parquet(IN_PATH)

# 2) ba_data.t_prdlst_cl 내려받기 — 읽기 전용 세션에서 SELECT 만
with _connect_readonly() as conn, conn.cursor() as cur:
    cur.execute("show default_transaction_read_only")
    assert cur.fetchone()[0] == "on", "읽기 전용 세션이 아닙니다"
    cur.execute("select prdlst_cd, kor_nm, lv::int as lv, hrnk_prdlst_cd from ba_data.t_prdlst_cl order by prdlst_cd")
    cl = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

# 3) 코드마다 상위 품목 코드(hrnk_prdlst_cd)를 따라 1단계(최상위)까지 올라가 lv1~lv6 코드·한글명을 만든다
_parent = dict(zip(cl["prdlst_cd"], cl["hrnk_prdlst_cd"]))
_kor    = dict(zip(cl["prdlst_cd"], cl["kor_nm"]))
_level  = dict(zip(cl["prdlst_cd"], cl["lv"]))
_max_lv = int(cl["lv"].max())


def _chain(code):
    chain, seen = {}, set()
    while code in _level and code not in seen:
        seen.add(code)
        chain[_level[code]] = code
        p = _parent.get(code)
        if not p or p == code:
            break
        code = p
    return chain


_rows = []
for code in cl["prdlst_cd"]:
    ch = _chain(code)
    rec = {"_join_key": code, "cl_kor_nm": _kor[code], "cl_lv": _level[code]}
    for l in range(1, _max_lv + 1):
        rec[f"lv{l}_cd"] = ch.get(l)
        rec[f"lv{l}_nm"] = _kor[ch[l]] if l in ch else None
    _rows.append(rec)
_tbl = pd.DataFrame(_rows)
_tbl["_join_key"] = _tbl["_join_key"].astype(str)

# 4) 왼쪽 조인 — 키 train_prdlst_code. 원자료 열은 그대로 두고 임시 키(_join_key)로 붙인 뒤 지운다
out = (src.assign(_join_key=src["train_prdlst_code"].astype(str))
          .merge(_tbl, on="_join_key", how="left", validate="many_to_one")
          .drop(columns="_join_key"))
out["cl_lv"] = out["cl_lv"].astype("Int64")   # 조인 안 된 행('기타' 170행)은 <NA>

# 4-1) 의뢰 시스템 코드·수입 LIMS 연결 여부 (7-1절과 같은 SELECT) — lims_data_key 로 왼쪽 조인
with _connect_readonly() as conn, conn.cursor() as cur:
    cur.execute("""
        select h.lims_data_key,
               m.reqest_sys_code,
               c.cmmn_code_nm as reqest_sys_nm,
               (s.rcept_no is not null) as import_lims_linked
        from ready_data.hazard_preprocessed_data h
        left join ba_data.m_org_lims_data_info m on m.lims_data_key = h.lims_data_key
        left join ba_data.t_cm_cmmn_code c on c.cmmn_cl_code = 'CI07' and c.cmmn_code = m.reqest_sys_code
        left join (select distinct rcept_no from ba_data.s_org_lims_data_info where label = '리스테리아') s
               on s.rcept_no = h.cntc_sys_key2
        where h.hazard_label = '리스테리아'
    """)
    _sys = pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])
assert _sys["lims_data_key"].is_unique, "lims_data_key 가 겹칩니다"
_aligned = _sys.set_index("lims_data_key").reindex(out["lims_data_key"].astype(str).to_numpy())   # 키 순서에 맞춘 값만 새 열로
assert _aligned["import_lims_linked"].notna().all(), "DB 에 없는 lims_data_key 가 있습니다"
out["reqest_sys_code"] = _aligned["reqest_sys_code"].to_numpy()
out["reqest_sys_nm"] = _aligned["reqest_sys_nm"].to_numpy()
out["import_lims_linked"] = _aligned["import_lims_linked"].to_numpy().astype(bool)

# 5) 점검 → 저장 → 다시 읽어 확인
assert len(out) == len(src), "조인 후 행 수가 달라졌습니다"
assert out[src.columns].equals(src), "원자료 열의 값 또는 행 순서가 바뀌었습니다"
out.to_parquet(OUT_PATH, index=False)

saved = pd.read_parquet(OUT_PATH)
n_pos = int((saved["judge_result"] == "검출").sum())
print(f"ba_data.t_prdlst_cl: {len(cl):,}행")
print(f"저장: {OUT_PATH} ({OUT_PATH.stat().st_size / 1024 ** 2:.1f} MB)")
print(f"행 {len(saved):,} / 열 {saved.shape[1]} / 검출 {n_pos:,}건 (검출률 {n_pos / len(saved) * 100:.3f}%)")
print(f"코드 매칭 {int(saved['cl_lv'].notna().sum()):,}행 / 매칭 안 됨 {int(saved['cl_lv'].isna().sum()):,}행")
print("의뢰 시스템 코드 이름별 행 수:", saved["reqest_sys_nm"].value_counts(dropna=False).to_dict())
print(f"수입 LIMS 연결 {int(saved['import_lims_linked'].sum()):,}행")
print("다시 읽은 파일 = 저장 전 결과:", saved.equals(out))
if "df_joined" in globals():
    print("위 1~5절 단계별 결과(df_joined)와 같음:", out.equals(df_joined))

ba_data.t_prdlst_cl: 8,864행
저장: C:\Users\NT751\Desktop\천영진\리스테리아ML\data\listeria_df_joined_full.parquet (46.3 MB)
행 676,537 / 열 40 / 검출 1,969건 (검출률 0.291%)
코드 매칭 676,367행 / 매칭 안 됨 170행
의뢰 시스템 코드 이름별 행 수: {'자체의뢰': 593955, '통합망': 35141, '수입축산물': 31583, '수입수산물': 7279, nan: 7203, '수입식품': 1376}
수입 LIMS 연결 47,441행
다시 읽은 파일 = 저장 전 결과: True
위 1~5절 단계별 결과(df_joined)와 같음: True
